In [3]:
# ============================================================
# IMPROVED OSTEOPOROSIS ARCHITECTURE
# Directional Difference Texture CNN (DDT-CNN)
# ============================================================

# FIXES:
# ✅ Huge flatten vector → AdaptiveAvgPooling
# ✅ Pure ANN → CNN architecture
# ✅ Weak spatial learning → Convolutions
# ✅ No hierarchical features → Multi Conv Blocks
# ✅ No normalization → BatchNorm
# ✅ No texture enhancement → CLAHE
# ✅ No attention → SE Attention Block
# ✅ No pooling issue → Average Pooling only
#
# ============================================================

import os
import cv2
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np

from PIL import Image

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    confusion_matrix,
    classification_report
)

# ============================================================
# SETTINGS
# ============================================================

DATASET_PATH = "total"

BATCH_SIZE = 32
EPOCHS = 10
IMG_SIZE = 128

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using Device:", device)

# ============================================================
# CLAHE TRANSFORM
# ============================================================

class CLAHETransform:
    def __call__(self, img):

        img = np.array(img)

        clahe = cv2.createCLAHE(
            clipLimit=2.0,
            tileGridSize=(8,8)
        )

        img = clahe.apply(img)

        return Image.fromarray(img)

# ============================================================
# TRANSFORMS
# ============================================================

transform = transforms.Compose([

    transforms.Grayscale(num_output_channels=1),

    CLAHETransform(),

    transforms.Resize((IMG_SIZE, IMG_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize((0.5,), (0.5,))
])

# ============================================================
# DATASET
# ============================================================

dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=transform
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size]
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ============================================================
# DIRECTIONAL DIFFERENCE FEATURE EXTRACTION
# ============================================================

def get_features(x):

    center = x[:, :, 1:-1, 1:-1]

    n1 = x[:, :, :-2, :-2]
    n2 = x[:, :, :-2, 1:-1]
    n3 = x[:, :, :-2, 2:]

    n4 = x[:, :, 1:-1, :-2]
    n5 = x[:, :, 1:-1, 2:]

    n6 = x[:, :, 2:, :-2]
    n7 = x[:, :, 2:, 1:-1]
    n8 = x[:, :, 2:, 2:]

    features = torch.cat([

        (center - n1),
        (center - n2),
        (center - n3),

        (center - n4),
        (center - n5),

        (center - n6),
        (center - n7),
        (center - n8)

    ], dim=1)

    # ========================================================
    # NORMALIZATION OF DIFFERENCE MAPS
    # ========================================================

    features = (features - features.mean()) / (
        features.std() + 1e-6
    )

    return features

# ============================================================
# SE ATTENTION BLOCK
# ============================================================

class SEBlock(nn.Module):

    def __init__(self, channels, reduction=8):
        super().__init__()

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(

            nn.Linear(channels, channels // reduction),

            nn.ReLU(),

            nn.Linear(channels // reduction, channels),

            nn.Sigmoid()
        )

    def forward(self, x):

        b, c, _, _ = x.size()

        y = self.pool(x).view(b, c)

        y = self.fc(y).view(b, c, 1, 1)

        return x * y

# ============================================================
# RESIDUAL BLOCK
# ============================================================

class ResidualBlock(nn.Module):

    def __init__(self, channels):
        super().__init__()

        self.block = nn.Sequential(

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(channels),

            nn.ReLU(),

            nn.Conv2d(
                channels,
                channels,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(channels)
        )

        self.relu = nn.ReLU()

    def forward(self, x):

        residual = x

        out = self.block(x)

        out += residual

        out = self.relu(out)

        return out

# ============================================================
# IMPROVED MODEL
# ============================================================

class OsteoNetDD(nn.Module):

    def __init__(self):
        super().__init__()

        # ====================================================
        # SHALLOW TEXTURE CNN
        # ====================================================

        self.features = nn.Sequential(

            # -----------------------------------------------
            # Block 1
            # -----------------------------------------------

            nn.Conv2d(
                8,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.AvgPool2d(2),

            # -----------------------------------------------
            # Block 2
            # -----------------------------------------------

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            ResidualBlock(64),

            SEBlock(64),

            nn.AvgPool2d(2),

            # -----------------------------------------------
            # Block 3
            # -----------------------------------------------

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            ResidualBlock(128),

            SEBlock(128),

            # =================================================
            # GLOBAL AVERAGE POOLING
            # =================================================

            nn.AdaptiveAvgPool2d((1,1))
        )

        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Dropout(0.4),

            nn.Linear(128, 64),

            nn.ReLU(),

            nn.Dropout(0.3),

            nn.Linear(64, 2)
        )

    def forward(self, x):

        # Directional Difference Maps
        x = get_features(x)

        # CNN Feature Learning
        x = self.features(x)

        # Classification
        x = self.classifier(x)

        return x

# ============================================================
# MODEL
# ============================================================

model = OsteoNetDD().to(device)

# ============================================================
# LOSS + OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# ============================================================
# TRAINING
# ============================================================

for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Loss: {total_loss:.4f}"
    )

# ============================================================
# EVALUATION
# ============================================================

model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (
            predicted == labels
        ).sum().item()

        all_preds.extend(
            predicted.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

# ============================================================
# RESULTS
# ============================================================

accuracy = correct / total

print("\n===================================")
print("OSTEONET-DD RESULTS")
print("===================================")

print(f"\nAccuracy: {accuracy:.4f}")

print("\n===================================")
print("CONFUSION MATRIX")
print("===================================")

cm = confusion_matrix(
    all_labels,
    all_preds
)

print(cm)

print("\n===================================")
print("CLASSIFICATION REPORT")
print("===================================")

print(
    classification_report(
        all_labels,
        all_preds
    )
)

Using Device: cpu
Epoch [1/10] Loss: 39.4580
Epoch [2/10] Loss: 35.7752
Epoch [3/10] Loss: 35.6533
Epoch [4/10] Loss: 35.7144
Epoch [5/10] Loss: 34.3378
Epoch [6/10] Loss: 34.4008
Epoch [7/10] Loss: 33.5488
Epoch [8/10] Loss: 33.5364
Epoch [9/10] Loss: 32.8398
Epoch [10/10] Loss: 32.7984

OSTEONET-DD RESULTS

Accuracy: 0.8135

CONFUSION MATRIX
[[257  98]
 [ 21 262]]

CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0       0.92      0.72      0.81       355
           1       0.73      0.93      0.81       283

    accuracy                           0.81       638
   macro avg       0.83      0.82      0.81       638
weighted avg       0.84      0.81      0.81       638



In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

# ===== SETTINGS =====
DATASET_PATH = "total"
BATCH_SIZE = 64
EPOCHS = 20
IMG_SIZE = 128
SEED = 42

torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ===== TRANSFORMS =====
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),

    # better normalization
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# ===== DATASET =====
dataset = datasets.ImageFolder(
    DATASET_PATH,
    transform=transform
)

train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size

train_dataset, test_dataset = torch.utils.data.random_split(
    dataset,
    [train_size, test_size],
    generator=torch.Generator().manual_seed(SEED)
)

# ===== DATALOADERS =====
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

# ===== FEATURE EXTRACTION =====
def get_features(x):

    center = x[:, :, 1:-1, 1:-1]

    n1 = x[:, :, :-2, :-2]
    n2 = x[:, :, :-2, 1:-1]
    n3 = x[:, :, :-2, 2:]

    n4 = x[:, :, 1:-1, :-2]
    n5 = x[:, :, 1:-1, 2:]

    n6 = x[:, :, 2:, :-2]
    n7 = x[:, :, 2:, 1:-1]
    n8 = x[:, :, 2:, 2:]

    features = torch.cat([
        (center - n1),
        (center - n2),
        (center - n3),
        (center - n4),
        (center - n5),
        (center - n6),
        (center - n7),
        (center - n8)
    ], dim=1)

    return features

# ===== PURE ANN MODEL =====
class PureANN(nn.Module):

    def __init__(self):
        super().__init__()

        input_size = 8 * 126 * 126

        self.fc = nn.Sequential(

            nn.Linear(input_size, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(512, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(128, 2)
        )

        self.initialize_weights()

    def initialize_weights(self):

        for m in self.modules():

            if isinstance(m, nn.Linear):

                nn.init.kaiming_normal_(
                    m.weight,
                    nonlinearity='relu'
                )

                nn.init.constant_(m.bias, 0)

    def forward(self, x):

        x = get_features(x)

        # safer than view()
        x = x.reshape(x.size(0), -1)

        x = self.fc(x)

        return x

# ===== MODEL =====
model = PureANN().to(device)

# ===== LOSS =====
criterion = nn.CrossEntropyLoss()

# ===== OPTIMIZER =====
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0005,
    weight_decay=1e-4
)

# ===== LR SCHEDULER =====
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    patience=2,
    factor=0.5
)

# ===== TRAINING =====
for epoch in range(EPOCHS):

    model.train()

    total_loss = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        loss = criterion(outputs, labels)

        optimizer.zero_grad()

        loss.backward()

        # gradient clipping
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)

    scheduler.step(avg_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {avg_loss:.4f}")

# ===== EVALUATION =====
model.eval()

correct = 0
total = 0

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

accuracy = correct / total

print(f"\nAccuracy: {accuracy:.4f}")

cm = confusion_matrix(all_labels, all_preds)

print("\nConfusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(all_labels, all_preds))

C:\Users\sunda\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch [1/20] Loss: 0.7414
Epoch [2/20] Loss: 0.3125
Epoch [3/20] Loss: 0.1429
Epoch [4/20] Loss: 0.0835
Epoch [5/20] Loss: 0.0521
Epoch [6/20] Loss: 0.0359
Epoch [7/20] Loss: 0.0194
Epoch [8/20] Loss: 0.0147
Epoch [9/20] Loss: 0.0154
Epoch [10/20] Loss: 0.0118
Epoch [11/20] Loss: 0.0099
Epoch [12/20] Loss: 0.0083
Epoch [13/20] Loss: 0.0057
Epoch [14/20] Loss: 0.0062
Epoch [15/20] Loss: 0.0057
Epoch [16/20] Loss: 0.0046
Epoch [17/20] Loss: 0.0083
Epoch [18/20] Loss: 0.0066
Epoch [19/20] Loss: 0.0051
Epoch [20/20] Loss: 0.0047

Accuracy: 0.6176

Confusion Matrix:
[[275  58]
 [186 119]]

Classification Report:
              precision    recall  f1-score   support

           0       0.60      0.83      0.69       333
           1       0.67      0.39      0.49       305

    accuracy                           0.62       638
   macro avg       0.63      0.61      0.59       638
weighted avg       0.63      0.62      0.60       638

